# The remaining OOP material that we did not cover in Session 5

--- 
# Decorators

Decorators in Python are functions that let you modify or extend the behavior of other functions or methods without changing their actual code. They are a powerful feature that makes code more reusable, clean, and expressive. 

You can think of decorators as functions applied on top of existing functions. We use the special character `@' to wrap a function with a decorator

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print("Before the function runs")
        result = func(*args, **kwargs)   # Call the original function
        print("After the function runs")
        return result
    return wrapper

@my_decorator
def say_hello(name):
    print(f"Hello, {name}!")

say_hello("Aitor")


### Special Python Decorators: @classmethod, @staticmethod, @property

| Decorator     | Purpose                                                                 | How It’s Called                          | Typical Use Case                                                                 | 
|---------------|-------------------------------------------------------------------------|------------------------------------------|----------------------------------------------------------------------------------|
| `@classmethod`| Defines a method that receives the **class** (`cls`) as the first argument instead of the instance (`self`). | Called on the class or instance.          | Factory methods, alternative constructors, or methods that need class-level data. |
| `@staticmethod`| Defines a method that does **not** receive `self` or `cls`. It behaves like a plain function inside the class namespace. | Called on the class or instance.          | Utility/helper functions that logically belong to the class but don’t need access to class/instance data. |
| `@property`   | Turns a method into a **read-only attribute**. Accessed like an attribute but executes code under the hood. | Called without parentheses (like an attribute). | Encapsulation: control access to private variables, computed attributes.          |


1.```@classmethod``` : Can be called with an instance of a class or directly by the class itself as its first argument. According to the Python documentation: It can be called either on the class (such as <code>C.f()</code>) or on an instance (such as <code>C().f()</code>). The instance is ignored except for its class. If a class method is called for a derived class, the derived class object is passed as the implied first argument. 

2.```@staticmethod``` : Is just a function inside of a class. You can call it both with and without instantiating the class. A typical use case is when you have a function where you believe it has a connection with a class. It’s a stylistic choice for the most part.

3.```@property``` : One of the simplest ways to use a property is to use it as a decorator of a method. This allows you to turn a class method into a class attribute. 


In [ ]:
class Car:
    total_cars = 0
    default_speed = 100

    def __init__(self, brand, speed=None):
        self.brand = brand
        self._speed = speed if speed is not None else Car.default_speed
        Car.total_cars += 1

    # Instance method
    def drive(self):
        return f"{self.brand} is driving at {self._speed} km/h"

    # Class method: count cars
    @classmethod
    def how_many_cars(cls):
        return f"There are {cls.total_cars} cars created."

    # Class method: alternative constructor
    @classmethod
    def from_string(cls, car_string):
        """Create a Car from a string like 'Tesla:150'"""
        brand, speed = car_string.split(":")
        return cls(brand, int(speed))

    # Static method
    @staticmethod
    def honk(): # Doesn't use any class data
        return "Beep beep!"

    # Property
    @property
    def speed(self):
        return self._speed




In [ ]:
# Normal constructor
c1 = Car("Toyota", 120)

# Alternative constructor via classmethod
c2 = Car.from_string("Tesla:150")

print(c1.drive())  
# Output: Toyota is driving at 120 km/h

print(c2.drive())  
# Output: Tesla is driving at 150 km/h

# Class method for counting
print(Car.how_many_cars())  
# Output: There are 2 cars created.

# Static method
print(Car.honk())  
# Output: Beep beep!

# Property
print(c2.speed)

### Exercise 10. Properties and protecting data

Make a Temperature class that stores Celsius internally, with a fahrenheit and kelvin properties that convert on the fly.

Use a setter so assigning to fahrenheit updates the Celsius value.

Reject temperatures below absolute zero.

Concepts: <code>@property</code>, setters, the _private naming convention.

## Back to some more relevant decorators
### The timer decorator is probably one of the most useful ones to measure your method execution time, when testing different approaches

In [ ]:
import functools
import time
def timer(func):
    """Print the runtime of the decorated function"""
    @functools.wraps(func)
    def wrapper_timer(*args, **kwargs):
        start_time = time.perf_counter()    # 1
        value = func(*args, **kwargs)
        end_time = time.perf_counter()      # 2
        run_time = end_time - start_time    # 3
        print(f"Finished {func.__name__!r} in {run_time:.4f} secs")
        return value
    return wrapper_timer

In [ ]:
from abc import ABC,abstractmethod
from scipy.stats import norm
import numpy as np

class Model(ABC):
    @abstractmethod
    def price_european_option(self,S0, K, T,CP,t=0):
        '''
        #Inputs:
        S0: initial stock price
        K: strike
        T: time to maturity
        CP: 1 for call, -1 for put
        '''
        pass

class Black_Scholes(Model):
    def __init__(self,sigma):
        self.sigma=sigma
    @timer
    def price_european_option(self,S0, K, CP, T,t=0):
        tau = T - t
        sigmtau = self.sigma*np.sqrt(tau)
        k = np.log(K/S0)
        dp = -k / sigmtau + 0.5*sigmtau
        dm = dp - sigmtau
        return S0*(CP*norm.cdf(CP*dp) - CP*np.exp(k)*norm.cdf(CP*dm))


In [ ]:
BS_model=Black_Scholes(sigma=0.2)
BS_model.price_european_option(S0=100,K=90,CP=-1,T=0.25)

# Decorators as classes

Decorators can also be classes! We need to define the ```__call__``` method, which is the one that will be called when defining it within a function. See an example below

In [ ]:
class DebugDecorator:
    """
    A decorator class for debugging instance methods.

    This decorator can be applied to methods of a class to print:
      - The instance's attributes (`self.__dict__`)
      - Positional arguments
      - Keyword arguments
      - The return value of the function

    It is useful for inspecting the state of an object, the inputs
    passed to its methods, and the output produced.

    Parameters
    ----------
    debug : bool
        Flag to enable or disable debugging output. If False, the
        decorated method runs normally without printing debug info.
    """

    def __init__(self, debug: bool):
        # Store whether debugging is enabled
        self.debug = debug

    def __call__(self, function):
        """
        Wrap the target function with debugging logic.

        Parameters
        ----------
        function : callable
            The method being decorated.

        Returns
        -------
        callable
            A wrapped version of the method that prints debug info
            if debugging is enabled.
        """
        def wrapper(function_self, *args, **kwargs):
            # Only print debug information if enabled
            if self.debug:
                print("[Instance attributes]")
                print(function_self.__dict__)

                if args:
                    print("[Positional arguments]")
                    print(args)

                if kwargs:
                    print("[Keyword arguments]")
                    print(kwargs)

            # Call the original method with its arguments
            result = function(function_self, *args, **kwargs)

            # Print the result if debugging is enabled
            if self.debug:
                print("[Return value]")
                print(result)

            return result

        return wrapper


### In this case the decorator needs to be called with the constructor, which will set the debug variable to either ```True``` or ```False```. This is useful to switch on and off decorators

In [ ]:
class MyClass:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    @DebugDecorator(debug=True)
    def add(self, z):
        return self.x + self.y + z

obj = MyClass(2, 3)
obj.add(5)

In [ ]:
from abc import ABC,abstractmethod
from scipy.stats import norm
import numpy as np

class Model(ABC):
    @abstractmethod
    def price_european_option(self,S0, K, sigma, T,CP,t=0):
        '''
        #Inputs:
        S0: initial stock price
        K: strike
        T: time to maturity
        CP: 1 for call, -1 for put
        '''
        pass

class Black_Scholes(Model):
    def __init__(self,sigma):
        self.sigma=sigma
    @DebugDecorator(debug=True)
    @timer
    def price_european_option(self,S0, K, CP, T,t=0):
        tau = T - t
        sigmtau = self.sigma*np.sqrt(tau)
        k = np.log(K/S0)
        dp = -k / sigmtau + 0.5*sigmtau
        dm = dp - sigmtau
        return S0*(CP*norm.cdf(CP*dp) - CP*np.exp(k)*norm.cdf(CP*dm))


In [ ]:
BS_model=Black_Scholes(sigma=0.2)
BS_model.price_european_option(S0=100,K=90,CP=-1,T=0.25)

### As you can see above we can also combine decorators! This make time measuring and debugging much cleaner and less error prone

# Debugging python (Restart kernel)
### You will soon notice that using ```print``` to debug, quickly starts to be very inefficient in involved projects. Python provides a simple debugger ```pdb``` out of the box

In [ ]:
def very_complex_function(x,y,z):
    intermediate_variable=x+y*z
    intermediate_variable2=0
    for i in range(10):
        intermediate_variable2+=intermediate_variable**i   
    return intermediate_variable2

In [ ]:
very_complex_function(1,2,-1)
#very_complex_function(1,0,0)

### perhaps not the solution we expected. An ideal debugging approach would be to be able to see the values of different variables as the function gets executed. These are called *breakpoints* and allow to break the code at a specific line. To do so Python has the method ```pdb.set_trace()```. Once we execute the code we can:
1- Type de name of the variable to see it's runtime value

2- Type ```n``` to move the execution of the code one line

3- Type ```c``` for the code to continue execution until next break point or termination

4- Type ```locals()``` to see all values of local variables

5- Type ```globals()``` to see all values of global variables

6- Type ```exit``` to exit the debugger

In [ ]:
import pdb

def very_complex_function(x,y,z):
    intermediate_variable=x+y*z
    intermediate_variable2=intermediate_variable**0.5
    pdb.set_trace()
    
    for i in range(10):
        intermediate_variable2+=intermediate_variable**i 
        pdb.set_trace()
    return intermediate_variable2

very_complex_function(1,2,-1)